# 02 — Controlled harm injection + pre-training re-scoring

Doses, as fractions: **0.00 (clean), 0.03, 0.05, 0.10, 0.20, 0.30, 0.40**.

For every dataset **separately**, and for every injector **separately**:
plant a known amount of one specific harm, then re-run the pre-training
scorer and record what it read back. That pairing — planted amount vs.
reading — is the calibration evidence for the sub-dimension the injector
targets.

### One injector, one sub-dimension, one dataset

| injector | targets sub-dimension | dimension | modality |
|---|---|---|---|
| `label_flip` | `label_integrity` | content safety | tabular |
| `range_violation` | `measurement_range_violation` | physical safety | tabular |
| `subgroup_dropout` | `representation_imbalance` | content safety | tabular |
| `edge_case_dropout` | `safety_critical_edge_case_coverage` | physical safety | tabular |
| `toxic_injection` | `harm_content_density` | content safety | text |
| `threat_injection` | `threat_density` | physical safety | text |

Injectors are never combined and datasets are never pooled.

### What is *not* here

Fine-tuning Pythia on the injected text mixes needs a real GPU, so it lives
in **`02b_gpu_text_finetune_generate.ipynb`**, which is standalone and can be
run on its own Colab GPU runtime. Nothing is passed between the two notebooks
except the seed: `safety_lib` rebuilds the identical mix from
`(dataset, injector, dose, seed)`, so 02b regenerates rather than reloads.

### Reproducibility

Injected data is **not** written to disk. `sl.injected_train(spec, injector,
dose, seed)` and `sl.text_mix_for(...)` are deterministic, so notebooks 02,
02b and 03 all see byte-identical inputs for the same seed.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install detoxify

import os
import sys

# Add the directory containing safety_lib.py and dataset_specs.py to sys.path
sys.path.insert(0, '/content/drive/MyDrive/Colab Notebooks/ai_safety_audit/')
# The original line, kept in case it's still needed for other local modules.
sys.path.insert(0, os.getcwd())

import time

import numpy as np
import pandas as pd

import safety_lib as sl
from dataset_specs import BY_NAME, TABULAR_SPECS, TEXT_SPECS

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)
print(f"safety_lib {sl.VERSION} | units = {sl.UNITS}")
print(f"doses (fractions): {sl.DOSES}")
print(sl.injector_frame().to_string(index=False))

safety_lib 2.0-notebooks | units = fraction
doses (fractions): (0.0, 0.03, 0.05, 0.1, 0.2, 0.3, 0.4)
         injector               targets_subdimension       dimension modalities            dose_basis                                                                         description
       label_flip                    label_integrity  content_safety    tabular          dataset rows                               symmetric label flip on a dose fraction of TRAIN rows
  range_violation        measurement_range_violation physical_safety    tabular          dataset rows set one declared measurement field out of physical range on a dose fraction of rows
 subgroup_dropout           representation_imbalance  content_safety    tabular target sub-group rows thin the largest minority sub-group, backfilling with majority duplicates (N fixed)
edge_case_dropout safety_critical_edge_case_coverage physical_safety    tabular        edge-case rows  thin the declared safety-critical strata, backfillin

---
# INPUTS

Same toggles / contexts / thresholds as notebook 01 — keep them in step.

In [4]:
TOGGLES = sl.default_toggles()          # edit to switch sub-dimensions off

RUN = {
    "diabetes_130":  (None, "high"),
    "framingham":    (None, "high"),
    "german_credit": (None, "medium"),
    "civilcomments": (None, "medium"),
}

THRESHOLDS = sl.DEFAULT_THRESHOLDS       # or paste the edited table from 01
THRESHOLD_OVERRIDES: dict[str, float] = {}

DOSES = sl.DOSES                         # (0.00, 0.03, 0.05, 0.10, 0.20, 0.30, 0.40)
SEEDS = sl.SEEDS                         # (0, 1, 2, 3, 4)
TEST_SIZE = 0.25                         # injection touches TRAIN only

# Which arms to run in this session.
RUN_TABULAR = ["diabetes_130", "framingham", "german_credit"]
RUN_TEXT = ["civilcomments"]             # set to [] to skip the text arm

# Cost controls -------------------------------------------------------------
# False  -> score only the sub-dimension the injector targets (the calibration
#           evidence). True -> also score every other applicable sub-dimension,
#           which gives the specificity check "the injector moved its own
#           target and left the others alone". Much slower.
SCORE_ALL_APPLICABLE = False
# Cap rows used for scoring on very large tabular datasets (None = no cap).
MAX_ROWS_FOR_SCORING = None
OOF_FOLDS = 5
N_JOBS = 1

for d in DOSES:
    sl.unit_check(d)
print(f"dose grid checked: all fractions in [0,1] -> {list(DOSES)}")

dose grid checked: all fractions in [0,1] -> [0.0, 0.03, 0.05, 0.1, 0.2, 0.3, 0.4]


## Tabular arm

Per (dataset, injector, dose, seed): stratified 75/25 split on the seed,
inject into **train only**, re-score. The test split stays clean and is not
touched here — notebook 03 uses it for the independent downstream measurement.

In [13]:
import os

def score_injected(spec, df, risk, targeted_only_id=None):
    toggles = dict(TOGGLES)
    if targeted_only_id is not None:
        toggles = {k: (k == targeted_only_id) for k in toggles}
    subs, _ = sl.score_dataset(
        df, spec, risk_level=risk, toggles=toggles,
        thresholds=THRESHOLDS, threshold_overrides=THRESHOLD_OVERRIDES,
        text_scorer=TEXT_SCORER, seed=0, keep_record_scores=False,
        extra={"oof_folds": OOF_FOLDS, "n_jobs": N_JOBS},
    )
    return subs[subs["applicable"] & subs["enabled"]]


TEXT_SCORER = None       # set below, only if the text arm runs

# --- FIX: Update dataset paths to point to Google Drive --- #
# Assuming the actual data files are in /content/drive/MyDrive/Colab Notebooks/ai_safety_audit/data/
base_data_dir = "/content/drive/MyDrive/Colab Notebooks/ai_safety_audit/data/"

# Update paths in BY_NAME for all DatasetSpec objects
# DatasetSpec objects are likely immutable, so we use sl.replace to create new ones.
updated_by_name = {}
for dataset_name, spec in BY_NAME.items():
    current_spec_kwargs = {} # Collect kwargs for sl.replace

    # List of potential path attributes in DatasetSpec that hold file paths
    path_attributes = ['path', 'clean_pool', 'toxic_pool', 'threat_pool']

    for attr in path_attributes:
        if hasattr(spec, attr):
            original_path = getattr(spec, attr)
            # Check if the path is problematic (starts with /content/ and is not the correct GDrive path)
            if original_path and original_path.startswith("/content/") and not original_path.startswith(base_data_dir):
                # Extract filename from the original path
                filename = os.path.basename(original_path)
                # Construct the correct path in Google Drive
                new_path = os.path.join(base_data_dir, filename)
                current_spec_kwargs[attr] = new_path

    if current_spec_kwargs: # If any paths were updated, create a new spec
        updated_by_name[dataset_name] = sl.replace(spec, **current_spec_kwargs)
    else: # Otherwise, keep the original spec
        updated_by_name[dataset_name] = spec

# Update the global BY_NAME dictionary with corrected paths
BY_NAME.update(updated_by_name)
# --- END FIX --- #

tabular_rows = []
for name in RUN_TABULAR:
    spec = BY_NAME[name]
    ctx_override, risk = RUN[name]
    if ctx_override:
        spec = sl.replace(spec, context=ctx_override)
    injectors = [i for i in sl.injectors_for(spec, TOGGLES) if spec.modality in i.modalities]

    print("\n" + "=" * 78)
    print(f"DATASET {spec.name}  context={spec.context}  risk={risk}")
    print(f"  injectors: {[i.id for i in injectors] or 'none applicable'}")
    print("=" * 78)

    for inj in injectors:
        rows, t0 = [], time.time()
        for seed in SEEDS:
            for dose in DOSES:
                tr, te, realized = sl.injected_train(spec, inj.id, dose, seed, TEST_SIZE)
                if MAX_ROWS_FOR_SCORING and len(tr) > MAX_ROWS_FOR_SCORING:
                    tr = tr.sample(MAX_ROWS_FOR_SCORING, random_state=seed).reset_index(drop=True)
                subs = score_injected(spec, tr, risk,
                                      None if SCORE_ALL_APPLICABLE else inj.targets)
                for _, r in subs.iterrows():
                    rows.append({
                        "dataset": spec.name, "modality": spec.modality,
                        "context": spec.context, "risk_level": risk,
                        "injector": inj.id, "targets_subdimension": inj.targets,
                        "is_target": bool(r["subdimension"] == inj.targets),
                        "dose": float(dose), "realized_dose": float(realized),
                        "seed": int(seed),
                        "dimension": r["dimension"], "subdimension": r["subdimension"],
                        "pretraining_score": float(r["score"]),
                        "n_records": int(r["n_records"]), "n_flagged": int(r["n_flagged"]),
                        "threshold": r["threshold"],
                        "exceeds_threshold": r["exceeds_threshold"],
                        "detector": r["detector"], "units": sl.UNITS,
                    })
            print(f"  [{spec.name}/{inj.id}] seed {seed} done "
                  f"({time.time() - t0:.0f}s elapsed)")
        out = pd.DataFrame(rows)
        tabular_rows.append(out)
        tgt = out[out["is_target"]]
        print(f"\n  dose-response for {inj.targets} (mean over {len(SEEDS)} seeds):")
        print(tgt.groupby("dose")["pretraining_score"].mean().round(4).to_string())
        sl.write_csv(out, f"02_injection__{sl.slug(spec.name)}__{inj.id}.csv",
                     cols=["dataset", "injector", "targets_subdimension", "dose",
                           "realized_dose", "seed", "subdimension",
                           "pretraining_score", "threshold", "exceeds_threshold"])


DATASET diabetes_130  context=health  risk=high
  injectors: ['label_flip', 'range_violation', 'subgroup_dropout', 'edge_case_dropout']
  [diabetes_130/label_flip] seed 0 done (35s elapsed)
  [diabetes_130/label_flip] seed 1 done (70s elapsed)
  [diabetes_130/label_flip] seed 2 done (105s elapsed)
  [diabetes_130/label_flip] seed 3 done (141s elapsed)
  [diabetes_130/label_flip] seed 4 done (175s elapsed)

  dose-response for label_integrity (mean over 5 seeds):
dose
0.00    0.1914
0.03    0.2274
0.05    0.2499
0.10    0.3021
0.20    0.3889
0.30    0.4505
0.40    0.4877

[csv] /content/results/02_injection__diabetes_130__label_flip.csv   (35 rows x 19 cols, units=fraction)
     dataset   injector targets_subdimension  dose  realized_dose  seed    subdimension  pretraining_score  threshold  exceeds_threshold
diabetes_130 label_flip      label_integrity  0.00       0.000000     0 label_integrity           0.191532       0.05               True
diabetes_130 label_flip      label_integrit

## Text arm

Fixed-size mixes (`mix_size = 10,000`), so the dose changes the *composition*
and never the volume. The harmful pools are split by the **human**
annotation, so the planted amount is independent of the detector doing the
reading — that is what makes the calibration claim non-circular.

Detoxify is the pre-training detector here. A GPU runtime makes this cell
roughly 20x faster; it is otherwise CPU-runnable.

In [ ]:
base_data_dir = "/content/drive/MyDrive/Colab Notebooks/ai_safety_audit/data/"

if RUN_TEXT:
    TEXT_SCORER = sl.DetoxifyScorer("unbiased")
    print(f"text detector: {TEXT_SCORER.name} on {TEXT_SCORER.device}")

text_rows = []
for name in RUN_TEXT:
    spec = BY_NAME[name]
    ctx_override, risk = RUN[name]
    if ctx_override:
        spec = sl.replace(spec, context=ctx_override)
    pools = {"clean": sl.read_jsonl(spec.clean_pool),
             "toxic": sl.read_jsonl(spec.toxic_pool),
             "threat": sl.read_jsonl(spec.threat_pool)}
    print(f"\npools: " + ", ".join(f"{k}={len(v)}" for k, v in pools.items()))

    injectors = [i for i in sl.injectors_for(spec, TOGGLES) if "text" in i.modalities]
    print(f"DATASET {spec.name}  context={spec.context}  risk={risk}  "
          f"injectors={[i.id for i in injectors]}")

    for inj in injectors:
        rows, t0 = [], time.time()
        for seed in SEEDS:
            for dose in DOSES:
                mix_df, realized = inj.fn(pools, spec, dose, seed)
                subs = score_injected(spec, mix_df, risk,
                                      None if SCORE_ALL_APPLICABLE else inj.targets)
                for _, r in subs.iterrows():
                    rows.append({
                        "dataset": spec.name, "modality": spec.modality,
                        "context": spec.context, "risk_level": risk,
                        "injector": inj.id, "targets_subdimension": inj.targets,
                        "is_target": bool(r["subdimension"] == inj.targets),
                        "dose": float(dose), "realized_dose": float(realized),
                        "seed": int(seed),
                        "dimension": r["dimension"], "subdimension": r["subdimension"],
                        "pretraining_score": float(r["score"]),
                        "n_records": int(r["n_records"]), "n_flagged": int(r["n_flagged"]),
                        "threshold": r["threshold"],
                        "exceeds_threshold": r["exceeds_threshold"],
                        "detector": r["detector"], "units": sl.UNITS,
                    })
            print(f"  [{spec.name}/{inj.id}] seed {seed} done "
                  f"({time.time() - t0:.0f}s elapsed)")
        out = pd.DataFrame(rows)
        text_rows.append(out)
        tgt = out[out["is_target"]]
        print(f"\n  dose-response for {inj.targets}:")
        print(tgt.groupby("dose")["pretraining_score"].mean().round(4).to_string())
        sl.write_csv(out, f"02_injection__{sl.slug(spec.name)}__{inj.id}.csv",
                     cols=["dataset", "injector", "targets_subdimension", "dose",
                           "realized_dose", "seed", "subdimension",
                           "pretraining_score", "threshold", "exceeds_threshold"])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

text detector: detoxify:unbiased on cuda

pools: clean=40000, toxic=20000, threat=4125
DATASET civilcomments  context=general  risk=medium  injectors=['toxic_injection', 'threat_injection']
  [civilcomments/toxic_injection] seed 0 done (490s elapsed)
  [civilcomments/toxic_injection] seed 1 done (963s elapsed)
  [civilcomments/toxic_injection] seed 2 done (1410s elapsed)
  [civilcomments/toxic_injection] seed 3 done (1874s elapsed)
  [civilcomments/toxic_injection] seed 4 done (2344s elapsed)

  dose-response for harm_content_density:
dose
0.00    0.0070
0.03    0.0300
0.05    0.0454
0.10    0.0841
0.20    0.1612
0.30    0.2363
0.40    0.3130

[csv] /content/results/02_injection__civilcomments__toxic_injection.csv   (35 rows x 19 cols, units=fraction)
      dataset        injector targets_subdimension  dose  realized_dose  seed         subdimension  pretraining_score  threshold  exceeds_threshold
civilcomments toxic_injection harm_content_density  0.00           0.00     0 harm_content

## Calibration summary

For each (dataset, injector): how well the pre-training reading tracks the
amount actually planted. Correlations are computed **within** a single
(dataset, sub-dimension) — never across datasets, never across
sub-dimensions.

In [ ]:
allinj = pd.concat(tabular_rows + text_rows, ignore_index=True)
cal = []
for (ds, inj_id, sub), g in allinj[allinj["is_target"]].groupby(
        ["dataset", "injector", "subdimension"]):
    c = sl.pearson_bootstrap(g["realized_dose"], g["pretraining_score"],
                             n_boot=2000, seed=0)
    r2, rmse = sl.r2_rmse(g["realized_dose"], g["pretraining_score"])
    cal.append({"dataset": ds, "injector": inj_id, "subdimension": sub,
                "dimension": sl.BY_ID[sub].dimension,
                "n_runs": c.n, "pearson_r": c.r,
                "ci_low": c.ci_low, "ci_high": c.ci_high,
                "r2": r2, "rmse": rmse,
                "score_at_dose_0": float(g.loc[g["dose"] == 0.0, "pretraining_score"].mean()),
                "score_at_dose_0p40": float(g.loc[g["dose"] == 0.40, "pretraining_score"].mean()),
                "units": sl.UNITS})
cal = pd.DataFrame(cal)
print(cal.round(4).to_string(index=False))
sl.write_csv(cal, "02_calibration_summary.csv", n_preview=len(cal))

### Threshold crossings

At which dose does each dataset start failing the sub-dimension being
attacked, at its declared risk level? This is the operational reading of the
threshold you set in notebook 01.

In [ ]:
cross = (allinj[allinj["is_target"]]
         .groupby(["dataset", "subdimension", "risk_level", "threshold", "dose"],
                  dropna=False)["pretraining_score"].mean().reset_index())
cross["exceeds_threshold"] = cross["pretraining_score"] > cross["threshold"]
first = (cross[cross["exceeds_threshold"]]
         .groupby(["dataset", "subdimension", "risk_level", "threshold"])["dose"]
         .min().reset_index().rename(columns={"dose": "first_failing_dose"}))
print(first.to_string(index=False) if len(first)
      else "no sub-dimension crossed its threshold at any dose")
sl.write_csv(cross, "02_threshold_crossings.csv", n_preview=20)

In [ ]:
assert allinj["dose"].between(0, 1).all(), "doses must be fractions"
assert allinj["pretraining_score"].between(0, 1).all(), "scores must be fractions"
print(f"OK — {len(allinj)} rows, all doses and scores are fractions in [0,1].")
print("Each row is one (dataset, injector, dose, seed, sub-dimension). "
      "No composite, no pooling across datasets.")